# Fixed-beta and rolling-beta tuning

Tune both hedge models using the exact validated pair book exported by notebook 05. This notebook uses days 501–700 only; days 701–1000 remain reserved for final testing.

## What is being compared?

**Fixed beta** calibrates beta and spread statistics once at the start of validation. **Rolling beta** refits them each day from a trailing window. The same pair book, validation period, limits, and score are used for both, so the comparison isolates the hedge-model choice.

In [49]:
from itertools import product
from pathlib import Path
import importlib.util
import sys

import numpy as np
import pandas as pd

DISCOVERY_END = 500
TUNING_END = 700

repo_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'backtester').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Run this notebook from inside the project directory.')
sys.path.insert(0, str(repo_root / 'backtester'))

from backtest.engine import run_backtest
from backtest.metrics import compute_metrics

price_path = repo_root / 'backtester' / 'data' / '2026' / 'prices.txt'
prices = pd.read_csv(price_path, sep=r'\s+', nrows=TUNING_END).to_numpy(dtype=float).T
book_path = repo_root / 'research_outputs' / 'validated_pair_book.csv'
if not book_path.exists():
    raise FileNotFoundError('Run every cell in notebooks/05_pairs_parameter_tuning.ipynb first.')
validated_book = pd.read_csv(book_path)
pair_book = tuple(validated_book[['ticker_a_index', 'ticker_b_index', 'ticker_a', 'ticker_b']].itertuples(index=False, name=None))

def load_strategy(module_name, filename):
    spec = importlib.util.spec_from_file_location(module_name, repo_root / 'src' / 'strategies' / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    module.PAIRS = pair_book
    module.reset_state()
    return module

fixed_strategy = load_strategy('validated_fixed_beta', 'pairs_12_fixed_beta.py')
rolling_strategy = load_strategy('validated_rolling_beta', 'pairs_12_rolling_beta.py')
print(f'{len(pair_book)} validated non-overlapping pairs; tuning days {DISCOVERY_END + 1}–{TUNING_END}.')

12 validated non-overlapping pairs; tuning days 501–700.


## 1. Fixed-beta parameter grid

Fixed beta varies its calibration lookback, entry/exit thresholds, and maximum holding period.

In [50]:
fixed_grid = {
    'lookback_days': [120, 180, 250],
    'entry_z': [0.50, 0.75, 1.00, 1.25, 1.50, 1.75],
    'exit_z': [0.00, 0.10, 0.25, 0.50],
    'max_holding_days': [0, 10, 20, 30, 45],  # 0 means no maximum hold
}

fixed_rows = []
for lookback_days, entry_z, exit_z, max_holding_days in product(*fixed_grid.values()):
    if exit_z >= entry_z:
        continue
    fixed_strategy.LOOKBACK_DAYS, fixed_strategy.ENTRY_Z = lookback_days, entry_z
    fixed_strategy.EXIT_Z, fixed_strategy.MAX_HOLDING_DAYS = exit_z, max_holding_days
    fixed_strategy.reset_state()
    metrics = compute_metrics(run_backtest(prices, fixed_strategy.getMyPosition, eval_start=DISCOVERY_END, eval_end=TUNING_END))
    fixed_rows.append({'model': 'fixed_beta', 'window_days': lookback_days, 'entry_z': entry_z, 'exit_z': exit_z, 'max_holding_days': max_holding_days, **{k: metrics[k] for k in ['score', 'mean_pl', 'ann_sharpe', 'max_drawdown', 'total_dvolume']}})
fixed_results = pd.DataFrame(fixed_rows).sort_values('score', ascending=False).reset_index(drop=True)
display(fixed_results.head(10).style.format({'score': '{:.2f}', 'mean_pl': '{:.2f}', 'ann_sharpe': '{:.2f}', 'max_drawdown': '{:.2f}', 'total_dvolume': '{:,.0f}'}).set_caption('Top fixed-beta settings'))

,model,window_days,entry_z,exit_z,max_holding_days,score,mean_pl,ann_sharpe,max_drawdown,total_dvolume
0,fixed_beta,180,0.500000,0.250000,0,329.39,334.66,7.91,-2698.04,"6,618,611"
1,fixed_beta,250,0.500000,0.100000,10,328.96,334.26,7.88,-2197.96,"9,176,700"
2,fixed_beta,180,0.500000,0.250000,45,328.91,334.20,7.88,-2698.04,"6,798,695"
3,fixed_beta,250,0.500000,0.000000,10,328.16,333.56,7.80,-2201.59,"8,803,239"
4,fixed_beta,250,0.500000,0.250000,10,327.92,333.11,7.95,-2329.28,"9,682,515"
5,fixed_beta,180,1.000000,0.500000,45,325.74,329.49,9.32,-1468.03,"5,054,421"
6,fixed_beta,180,1.000000,0.500000,0,324.23,328.01,9.26,-1468.03,"4,967,572"
7,fixed_beta,180,0.500000,0.250000,10,322.52,327.59,7.97,-2386.62,"9,648,048"
8,fixed_beta,180,0.500000,0.250000,20,322.30,327.69,7.73,-2698.04,"7,650,425"
9,fixed_beta,180,0.500000,0.250000,30,320.05,325.54,7.64,-2698.04,"7,176,448"


## 2. Rolling-beta parameter grid

Rolling beta uses the same thresholds and holding-period grid, but its window is the daily refit window. Shorter windows adapt faster and can be noisier.

In [51]:
rolling_grid = {
    'rolling_window_days': [120, 180, 250],
    'entry_z': [0.50, 0.75, 1.00, 1.25, 1.50, 1.75],
    'exit_z': [0.00, 0.10, 0.25, 0.50],
    'max_holding_days': [0, 10, 20, 30, 45],  # 0 means no maximum hold
}
rolling_rows = []
for rolling_window_days, entry_z, exit_z, max_holding_days in product(
    rolling_grid['rolling_window_days'],
    rolling_grid['entry_z'],
    rolling_grid['exit_z'],
    rolling_grid['max_holding_days'],
):
    if exit_z >= entry_z:
        continue
    rolling_strategy.ROLLING_WINDOW_DAYS = rolling_window_days
    rolling_strategy.ENTRY_Z = entry_z
    rolling_strategy.EXIT_Z = exit_z
    rolling_strategy.MAX_HOLDING_DAYS = max_holding_days
    rolling_strategy.reset_state()

    result = run_backtest(
        prices,
        rolling_strategy.getMyPosition,
        eval_start=DISCOVERY_END,
        eval_end=TUNING_END,
    )
    metrics = compute_metrics(result)
    rolling_rows.append({
        'model': 'rolling_beta', 'window_days': rolling_window_days,
        'entry_z': entry_z,
        'exit_z': exit_z,
        'max_holding_days': max_holding_days,
        'score': metrics['score'],
        'mean_pl': metrics['mean_pl'],
        'ann_sharpe': metrics['ann_sharpe'],
        'max_drawdown': metrics['max_drawdown'],
        'total_dvolume': metrics['total_dvolume'],
    })

rolling_results = pd.DataFrame(rolling_rows).sort_values('score', ascending=False).reset_index(drop=True)
display(rolling_results.head(10).style.format({
    'window_days': '{:.0f}', 'entry_z': '{:.2f}', 'exit_z': '{:.2f}',
    'max_holding_days': '{:.0f}', 'score': '{:.2f}', 'mean_pl': '{:.2f}',
    'ann_sharpe': '{:.2f}', 'max_drawdown': '{:.2f}', 'total_dvolume': '{:,.0f}',
}))

,model,window_days,entry_z,exit_z,max_holding_days,score,mean_pl,ann_sharpe,max_drawdown,total_dvolume
0,rolling_beta,250,0.50,0.10,0,353.63,359.25,7.94,-2601.62,"6,540,040"
1,rolling_beta,250,0.50,0.10,45,351.07,356.74,7.87,-2601.62,"6,659,568"
2,rolling_beta,250,0.50,0.10,20,349.14,354.82,7.84,-2817.42,"7,301,616"
3,rolling_beta,250,0.50,0.25,0,344.69,350.12,7.96,-2886.06,"7,296,637"
4,rolling_beta,250,0.50,0.25,45,342.28,347.78,7.89,-2886.06,"7,353,330"
5,rolling_beta,250,0.75,0.50,0,341.95,346.01,9.18,-1918.91,"7,331,620"
6,rolling_beta,250,0.75,0.50,45,341.95,346.01,9.18,-1918.91,"7,331,620"
7,rolling_beta,250,0.50,0.10,30,341.14,347.07,7.58,-2725.42,"6,771,579"
8,rolling_beta,250,0.75,0.50,30,340.70,344.75,9.17,-1918.91,"7,413,190"
9,rolling_beta,250,0.50,0.25,20,340.31,345.69,7.95,-3101.86,"7,934,529"


## 3. Compare the two hedge models

Compare the best configuration from each model on the same validation period. Choose the model that is strong and reasonably robust across nearby settings; do not choose based on the final test.

In [52]:
model_comparison = pd.concat([fixed_results.head(1), rolling_results.head(1)], ignore_index=True).sort_values('score', ascending=False)
display(model_comparison.style.format({
    'window_days': '{:.0f}', 'entry_z': '{:.2f}', 'exit_z': '{:.2f}',
    'max_holding_days': '{:.0f}', 'score': '{:.2f}', 'mean_pl': '{:.2f}',
    'ann_sharpe': '{:.2f}', 'max_drawdown': '{:.2f}', 'total_dvolume': '{:,.0f}',
}).set_caption('Best validation configuration for each hedge model'))
print('Freeze one row above, update its corresponding strategy file, then evaluate it once on days 701–1000.')

,model,window_days,entry_z,exit_z,max_holding_days,score,mean_pl,ann_sharpe,max_drawdown,total_dvolume
1,rolling_beta,250,0.50,0.10,0,353.63,359.25,7.94,-2601.62,"6,540,040"
0,fixed_beta,180,0.50,0.25,0,329.39,334.66,7.91,-2698.04,"6,618,611"


Freeze one row above, update its corresponding strategy file, then evaluate it once on days 701–1000.
